# Session 08: pandas II: cleaning the mess

Clean the dirty export of Lakeside Mart order lines. Each task has a check. At the end the cleaned net revenue must reconcile to the till system's total, with a named residual.

**Student version.** Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("orders.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

## Task 1. Load and keep the raw file
Read `orders_raw.csv` into `raw` and print its shape. Make a copy called `df` to clean. The `note` function that fills the cleaning log is given to you.

In [ ]:
import pandas as pd

# Your turn: read orders_raw.csv into raw and print its shape
# Then make a copy of raw called df, and clean df only

log = []
def note(step, before, after, why):
    log.append({'step': step, 'rows_before': before, 'rows_after': after, 'why': why})


In [ ]:
_missing = [v for v in ('df', 'raw',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert raw.shape == (41259, 10), 'Check the file name'
assert len(df) == 41259, 'df should be a copy of raw'
print('Task 1 correct')

## Task 2. Duplicates
Drop the exact duplicate lines from `df`. Store the number removed in `removed`, print it, and log the step.

In [ ]:
# Your turn: remember the row count in before, then drop the exact duplicate lines from df
# Store the number removed in removed and print it
# Log the step with note(...)


In [ ]:
_missing = [v for v in ('df', 'removed',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert removed == 80, 'Use df.drop_duplicates() and compare the lengths'
assert df.line_id.is_unique, 'line_id should now appear once each'
print('Task 2 correct')

## Task 3. Prices and dates
Convert `unit_price` to whole numbers (remove `UGX` and commas first). Then convert `order_date` to real dates: dashes are year-month-day, slashes are day/month/year. Log both steps.

In [ ]:
# Your turn, prices: take the text UGX and the commas out of unit_price, then make it whole numbers

# Your turn, dates: some dates are year-month-day and some are day/month/year
# Read the column once with each format, so each reading is blank where it does not fit,
# then fill the blanks of the first reading from the second

# Log both steps with note(...)


In [ ]:
_missing = [v for v in ('df',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert df.unit_price.dtype == 'int64', 'unit_price should be whole numbers'
assert df.order_date.isna().sum() == 0, 'Every date should have one valid reading'
assert df.order_date.min() == pd.Timestamp('2024-01-01'), 'Earliest date is 1 January 2024'
assert df.order_date.max() == pd.Timestamp('2025-12-28'), 'Latest date is 28 December 2025'
print('Task 3 correct')

## Task 4. Branch and payment text
Trim, fix the case and fix the two typos in `branch` (`Kampala Cntral`, `Entebe`). Make `payment` one of `Mobile money`, `Cash`, `Card`. Log the step.

In [ ]:
# Your turn: in branch, remove stray spaces, fix the capital letters, then correct the two typos
# Your turn: in payment, remove stray spaces, use lower case, change 'mm' to 'mobile money', then capitalise the first letter
# Log the step with note(...)


In [ ]:
_missing = [v for v in ('df',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert df.branch.nunique() == 8, 'There are eight branches'
assert sorted(df.payment.unique()) == ['Card', 'Cash', 'Mobile money'], 'Three payment methods'
print('Task 4 correct')

## Task 5. Missing customers and missing quantities
Fill every blank customer with `walk-in` (read_csv already made some of them empty). Then drop the lines with no quantity, keeping them in `lost`. Log both steps.

In [ ]:
# Your turn: fill the blank customer_id values with 'walk-in', then count the walk-in lines into n_walkin
# Your turn: keep the lines with no qty in lost, then drop them from df
# Log both steps with note(...)


In [ ]:
_missing = [v for v in ('df', 'lost', 'n_walkin',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert n_walkin == 12456, 'Fill customer_id with walk-in and count it'
assert len(lost) == 12, 'Twelve lines have no quantity'
assert len(df) == 41167, 'Drop only the blank quantities, using subset'
print('Task 5 correct')

## Task 6. Outliers: check before you remove
Count the lines whose price differs from `products.csv` (`n_mismatch`), and the lines with a negative quantity (`n_returns`). Remove nothing, and log that.

In [ ]:
# Your turn: join df to products.csv on product_id
# n_mismatch = the number of lines whose unit_price differs from the price list unit_price_ugx
# n_returns = the number of lines with a negative qty
# Remove nothing. Log the step with note(...)


In [ ]:
_missing = [v for v in ('df', 'n_mismatch', 'n_returns',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert n_mismatch == 0, 'Every price should match the price list'
assert n_returns == 1179, 'Count qty below zero'
assert len(df) == 41167, 'Nothing should be removed in this task'
print('Task 6 correct')

## Task 7. Reconcile
Add `revenue = qty * unit_price * (1 - discount_pct / 100)` and sum it into `net`. The till system's own total for the same lines is 561,421,125 UGX, stored as `true_net`. Compute `residual = true_net - net`, then print the cleaning log.

In [ ]:
true_net = 561421125

# Your turn: add a revenue column, sum it into net and round to a whole number
# Your turn: work out residual from true_net and net, then print both
# Make the last line of the cell the log as a table


In [ ]:
_missing = [v for v in ('log', 'net', 'residual',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert net == 561288435, 'Check the revenue formula and that returns are kept'
assert residual == 132690, 'residual = true_net - net'
assert len(log) == 7, 'The log should have 7 rows'
assert log[-1]['rows_after'] == 41167, 'Last row count should be 41167'
print('Task 7 correct')

## Task 8. Name the residual
In one sentence in this cell, say what the residual is and what causes it. Then save your work with `df.to_csv('orders_clean.csv', index=False)` and write one line on how you would explain the cleaning log to a manager. This notebook is Checkpoint 1: it passes when every check passes and the residual is named.